In [7]:
import numpy as np

from env.demand_generator import DemandGenerator
gen = DemandGenerator(n_skus=20)
for _ in range(5):
    order = gen.sample_order()
    print(f"Order: {list(np.where(order)[0])}")


Order: [np.int64(9)]
Order: [np.int64(2), np.int64(4)]
Order: [np.int64(5), np.int64(7)]
Order: [np.int64(1), np.int64(5), np.int64(13), np.int64(19)]
Order: [np.int64(0)]


In [8]:
import sys
sys.modules.pop("env.warehouse_env", None)

from env.warehouse_env import WarehouseEnv

In [9]:
from env.warehouse_env import WarehouseEnv
env = WarehouseEnv(grid_size=5, n_skus=20, orders_per_episode=50, reward_mode="dense", max_swaps_per_order=10, seed=42)
obs, info = env.reset()
print("Obs shape:", obs.shape)      # should be (45,) = 25 slots + 20 skus
print("Action space:", env.action_space.n)  # should be 625
env.render()


for _ in range(5):
    action = env.action_space.sample()
    obs, reward, terminated, truncated, info = env.step(action)
    print(f"Action: {action}, Reward: {reward:.2f}")


print("✅ Environment working")


Obs shape: (45,)
Action space: 625

Order: [np.int64(9)]
Grid (SKU IDs, -1=empty, depot=top-left):
 15  16  19   -   9
  7  17   -   6  10
  3   0   -  18  12
  5  11  14   -   2
  4   -   1  13   8
Travel dist: 8.0
Action: 424, Reward: 0.00
Action: 109, Reward: -2.00
Action: 344, Reward: 0.00
Action: 514, Reward: 0.00
Action: 381, Reward: 0.00
✅ Environment working


In [10]:
from gymnasium.envs.registration import register

register(
    id="WarehouseSlotting-v0",
    entry_point="env.warehouse_env:WarehouseEnv",
    max_episode_steps=500,
)

In [11]:
import gymnasium as gym
import env  # triggers __init__.py

env_instance = gym.make("WarehouseSlotting-v0")
print(env_instance.observation_space)  # Box(-1.0, 20.0, (45,), float32)
print("Registration OK")

Box(-1.0, 20.0, (45,), float32)
Registration OK


In [12]:
from env.warehouse_env import WarehouseEnv
from agents.random_agent import RandomAgent
from agents.heuristic_agent import FrequencyHeuristicAgent

env = WarehouseEnv()

random_agent = RandomAgent(env.action_space)
heuristic_agent = FrequencyHeuristicAgent(env)

for name, agent in [("Random", random_agent), ("Heuristic", heuristic_agent)]:
    obs, _ = env.reset()
    total_travel = 0
    done = False

    while not done:
        action, _ = agent.predict(obs)
        obs, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated

    print(f"{name}: total travel = {info['episode_travel']:.1f}")

Random: total travel = 1034.0
Heuristic: total travel = 884.0


In [13]:
def evaluate(agent, env, episodes=100):
    total = 0
    for _ in range(episodes):
        obs, _ = env.reset()
        done = False
        while not done:
            action, _ = agent.predict(obs)
            obs, reward, terminated, truncated, info = env.step(action)
            done = terminated or truncated
        total += info["episode_travel"]
    return total / episodes

print("Random avg:", evaluate(random_agent, env))
print("Heuristic avg:", evaluate(heuristic_agent, env))

Random avg: 989.42
Heuristic avg: 823.06


In [14]:
import os
os.environ["WANDB_START_METHOD"] = "thread"
os.environ["WANDB_DISABLE_SERVICE"] = "true"

In [19]:
import gymnasium as gym
import yaml
import os
import wandb
import env  # registers WarehouseSlotting-v0
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.callbacks import EvalCallback, BaseCallback
from stable_baselines3.common.monitor import Monitor

class WandbCallback(BaseCallback):
    """Log SB3 metrics to W&B at each rollout."""
    def __init__(self, verbose=0):
        super().__init__(verbose)

    def _on_step(self):
        return True

    def _on_rollout_end(self):
        if len(self.model.ep_info_buffer) > 0:
            ep_rewards = [ep["r"] for ep in self.model.ep_info_buffer]
            wandb.log({
                "rollout/ep_rew_mean": sum(ep_rewards) / len(ep_rewards),
                "rollout/n_updates": self.model.num_timesteps,
            })
        return True


def train(config_path: str = "configs/default.yaml"):
    with open(config_path) as f:
        cfg = yaml.safe_load(f)

    run_name = cfg["training"]["run_name"]

    # W&B init
    wandb.init(
        project="warehouse-rl",
        name=run_name,
        config=cfg,
        sync_tensorboard=True
    )

    os.makedirs(cfg["training"]["log_dir"], exist_ok=True)
    os.makedirs(cfg["training"]["model_dir"], exist_ok=True)

    # Vectorised envs (4 parallel) for faster data collection
    env_kwargs = cfg["env"]
    train_env = make_vec_env(
        "WarehouseSlotting-v0",
        n_envs=4,
        env_kwargs=env_kwargs
    )

    # Eval env (single, not vectorised)
    eval_env = Monitor(gym.make("WarehouseSlotting-v0", **env_kwargs))

    # Eval callback: saves best model automatically
    eval_callback = EvalCallback(
        eval_env,
        best_model_save_path=cfg["training"]["model_dir"],
        log_path=cfg["training"]["log_dir"],
        eval_freq=cfg["training"]["eval_freq"],
        n_eval_episodes=cfg["training"]["n_eval_episodes"],
        deterministic=True,
        render=False
    )

    # PPO model
    model = PPO(
        policy=cfg["ppo"]["policy"],
        env=train_env,
        n_steps=cfg["ppo"]["n_steps"],
        batch_size=cfg["ppo"]["batch_size"],
        n_epochs=cfg["ppo"]["n_epochs"],
        learning_rate=cfg["ppo"]["learning_rate"],
        gamma=cfg["ppo"]["gamma"],
        gae_lambda=cfg["ppo"]["gae_lambda"],
        clip_range=cfg["ppo"]["clip_range"],
        ent_coef=cfg["ppo"]["ent_coef"],
        vf_coef=cfg["ppo"]["vf_coef"],
        max_grad_norm=cfg["ppo"]["max_grad_norm"],
        tensorboard_log=cfg["training"]["log_dir"],
        verbose=1
    )

    print(f"\n🚀 Starting PPO training: {run_name}")
    print(f"   Timesteps: {cfg['ppo']['total_timesteps']:,}")
    print(f"   Reward mode: {env_kwargs['reward_mode']}\n")

    model.learn(
        total_timesteps=cfg["ppo"]["total_timesteps"],
        callback=[eval_callback, WandbCallback()],
        progress_bar=True
    )

    model.save(os.path.join(cfg["training"]["model_dir"], f"{run_name}_final"))
    print(f"\n✅ Training complete. Model saved.")
    wandb.finish()


if __name__ == "__main__":
    train()

ServicePollForTokenError: Failed to read port info after 30.0 seconds.

In [23]:
import wandb
wandb.init(mode="offline")
print("success")

ServicePollForTokenError: Failed to read port info after 30.0 seconds.